# Preprocessing

In [2]:
# Load engineered data and persisted project artifacts

import json
import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    "../data/engineered/ronaldo_1000_ml_engineered.csv"
)

ARTIFACT_DIR = Path("../artifacts")

FEATURE_SPEC_PATH = ARTIFACT_DIR / "feature_spec.json"
TEMPORAL_FOLDS_PATH = ARTIFACT_DIR / "temporal_folds.json"
VALIDATION_POLICY_PATH = ARTIFACT_DIR / "validation_policy.json"


# Load engineered dataset
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

df = df.sort_values("date").reset_index(drop=True)


# Load feature specification
with open(
    FEATURE_SPEC_PATH,
    "r",
    encoding="utf-8"
) as f:
    feature_spec = json.load(f)


# Load temporal validation design
with open(
    TEMPORAL_FOLDS_PATH,
    "r",
    encoding="utf-8"
) as f:
    temporal_folds_spec = json.load(f)


# Load validation policy
with open(
    VALIDATION_POLICY_PATH,
    "r",
    encoding="utf-8"
) as f:
    validation_policy = json.load(f)


FULL_FEATURES = feature_spec["full_features"]
NUMERIC_FEATURES = feature_spec["numeric_features"]
CATEGORICAL_FEATURES = feature_spec["categorical_features"]

TARGET = feature_spec["primary_target"]
SECONDARY_TARGETS = feature_spec["secondary_targets"]


# Integrity checks
assert len(df) == 1333
assert df["match_id"].nunique() == 1333
assert df[TARGET].sum() == 978

assert len(FULL_FEATURES) == 84
assert len(NUMERIC_FEATURES) == 77
assert len(CATEGORICAL_FEATURES) == 7

assert set(FULL_FEATURES).issubset(df.columns)
assert df[FULL_FEATURES].isna().sum().sum() == 0

assert validation_policy["primary_target"] == TARGET
assert validation_policy["final_test_start"] == "2025-01-01"
assert len(temporal_folds_spec) == 4


print("PREPROCESSING PROJECT LOAD")
print("=" * 60)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min().date(), "→", df["date"].max().date())

print("\nFeature contract:")
print("Full features:", len(FULL_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))

print("\nValidation design:")
print("Temporal folds:", len(temporal_folds_spec))
print(
    "Primary metric:",
    validation_policy["primary_metric"]
)
print(
    "Final test start:",
    validation_policy["final_test_start"]
)

print("\nArtifact consistency: PASSED")

PREPROCESSING PROJECT LOAD
Dataset shape: (1333, 100)
Date range: 2002-08-14 → 2026-08-28

Feature contract:
Full features: 84
Numeric features: 77
Categorical features: 7

Validation design:
Temporal folds: 4
Primary metric: poisson_deviance
Final test start: 2025-01-01

Artifact consistency: PASSED


## Audit unseen categories fold by fold

In [7]:
# Audit categorical generalization across temporal folds

import numpy as np

categorical_audit_rows = []

for fold in temporal_folds_spec:
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])

    val_start = pd.Timestamp(fold["validation_start"])
    val_end = pd.Timestamp(fold["validation_end"])

    train = df[
        (df["date"] >= train_start)
        & (df["date"] <= train_end)
    ]

    val = df[
        (df["date"] >= val_start)
        & (df["date"] <= val_end)
    ]

    for col in CATEGORICAL_FEATURES:
        train_values = set(train[col].dropna().unique())
        val_values = set(val[col].dropna().unique())

        unseen_values = val_values - train_values

        unseen_mask = val[col].isin(unseen_values)

        categorical_audit_rows.append({
            "fold": fold["name"],
            "feature": col,
            "train_unique": len(train_values),
            "val_unique": len(val_values),
            "unseen_categories": len(unseen_values),
            "val_rows_unseen": int(unseen_mask.sum()),
            "unseen_row_pct": (
                unseen_mask.mean() * 100
                if len(val) > 0
                else np.nan
            ),
        })


categorical_audit = pd.DataFrame(
    categorical_audit_rows
)


print("UNSEEN CATEGORY AUDIT BY TEMPORAL FOLD")
print("=" * 100)

print(
    categorical_audit
    .round({
        "unseen_row_pct": 2
    })
    .to_string(index=False)
)


print("\nSUMMARY — TOTAL UNSEEN VALIDATION ROWS BY FEATURE")
print("=" * 70)

summary = (
    categorical_audit
    .groupby("feature", as_index=False)
    .agg(
        folds_with_unseen=("unseen_categories", lambda x: (x > 0).sum()),
        total_unseen_categories=("unseen_categories", "sum"),
        total_val_rows_unseen=("val_rows_unseen", "sum"),
    )
    .sort_values(
        "total_val_rows_unseen",
        ascending=False
    )
)

print(summary.to_string(index=False))

UNSEEN CATEGORY AUDIT BY TEMPORAL FOLD
  fold          feature  train_unique  val_unique  unseen_categories  val_rows_unseen  unseen_row_pct
Fold 1 competition_type             4           4                  0                0            0.00
Fold 1            venue             3           2                  0                0            0.00
Fold 1      day_of_week             7           7                  0                0            0.00
Fold 1           season            16           3                  3              158          100.00
Fold 1             team             4           2                  1              133           84.18
Fold 1         opponent           203          52                 25               89           56.33
Fold 1      competition            20           9                  4              116           73.42
Fold 2 competition_type             4           4                  0                0            0.00
Fold 2            venue             3      

## Define preprocessing feature sets

In [10]:
# Define model feature-set variants

IDENTITY_CATEGORICALS = feature_spec["identity_categoricals"]
STABLE_CATEGORICALS = feature_spec["stable_categoricals"]


# 1. Full feature set
FULL_MODEL_FEATURES = FULL_FEATURES.copy()


# 2. Context/generalization feature set
# Remove high-cardinality / future-unseen identity categories.
CONTEXT_MODEL_FEATURES = [
    feature
    for feature in FULL_FEATURES
    if feature not in IDENTITY_CATEGORICALS
]


# 3. Intermediate feature set
# Keep team/opponent/competition identity, but remove literal season.
NO_SEASON_MODEL_FEATURES = [
    feature
    for feature in FULL_FEATURES
    if feature != "season"
]


FEATURE_SETS = {
    "full": FULL_MODEL_FEATURES,
    "context": CONTEXT_MODEL_FEATURES,
    "no_season": NO_SEASON_MODEL_FEATURES,
}


print("MODEL FEATURE SETS")
print("=" * 70)

for name, features in FEATURE_SETS.items():
    categorical = [
        feature
        for feature in features
        if feature in CATEGORICAL_FEATURES
    ]

    numeric = [
        feature
        for feature in features
        if feature in NUMERIC_FEATURES
    ]

    print(f"\n{name.upper()}")
    print("-" * 40)
    print("Total features:", len(features))
    print("Numeric:", len(numeric))
    print("Categorical:", len(categorical))
    print("Categoricals:", categorical)


# Integrity checks
assert len(FULL_MODEL_FEATURES) == 84
assert len(CONTEXT_MODEL_FEATURES) == 80
assert len(NO_SEASON_MODEL_FEATURES) == 83

assert set(STABLE_CATEGORICALS).issubset(
    CONTEXT_MODEL_FEATURES
)

assert not set(IDENTITY_CATEGORICALS).intersection(
    CONTEXT_MODEL_FEATURES
)

assert "season" not in NO_SEASON_MODEL_FEATURES

assert {
    "team",
    "opponent",
    "competition"
}.issubset(NO_SEASON_MODEL_FEATURES)

print("\nFeature-set integrity: PASSED")

MODEL FEATURE SETS

FULL
----------------------------------------
Total features: 84
Numeric: 77
Categorical: 7
Categoricals: ['season', 'team', 'opponent', 'competition', 'competition_type', 'venue', 'day_of_week']

CONTEXT
----------------------------------------
Total features: 80
Numeric: 77
Categorical: 3
Categoricals: ['competition_type', 'venue', 'day_of_week']

NO_SEASON
----------------------------------------
Total features: 83
Numeric: 77
Categorical: 6
Categoricals: ['team', 'opponent', 'competition', 'competition_type', 'venue', 'day_of_week']

Feature-set integrity: PASSED


In [12]:
# Build leakage-safe preprocessing factory

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


def build_linear_preprocessor(feature_set):
    """
    Create preprocessing for linear/count models.

    Important:
    This function only constructs the transformer.
    It does not fit it to any data.
    """

    numeric_features = [
        feature
        for feature in feature_set
        if feature in NUMERIC_FEATURES
    ]

    categorical_features = [
        feature
        for feature in feature_set
        if feature in CATEGORICAL_FEATURES
    ]

    numeric_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features,
            ),
        ],
        remainder="drop",
    )

    return (
        preprocessor,
        numeric_features,
        categorical_features,
    )


# Construct one preprocessor for each feature-set variant.
# Nothing is fitted yet.

PREPROCESSORS = {}

for name, features in FEATURE_SETS.items():

    preprocessor, numeric, categorical = (
        build_linear_preprocessor(features)
    )

    PREPROCESSORS[name] = {
        "preprocessor": preprocessor,
        "numeric_features": numeric,
        "categorical_features": categorical,
    }


print("LINEAR / COUNT PREPROCESSORS")
print("=" * 70)

for name, config in PREPROCESSORS.items():

    print(f"\n{name.upper()}")
    print("-" * 40)
    print(
        "Numeric inputs:",
        len(config["numeric_features"])
    )
    print(
        "Categorical inputs:",
        len(config["categorical_features"])
    )
    print(
        "Fitted:",
        hasattr(
            config["preprocessor"],
            "transformers_"
        )
    )


# Integrity checks
assert len(
    PREPROCESSORS["full"]["numeric_features"]
) == 77

assert len(
    PREPROCESSORS["full"]["categorical_features"]
) == 7

assert len(
    PREPROCESSORS["context"]["categorical_features"]
) == 3

assert len(
    PREPROCESSORS["no_season"]["categorical_features"]
) == 6

# Critical check: construction must not fit anything.
for config in PREPROCESSORS.values():
    assert not hasattr(
        config["preprocessor"],
        "transformers_"
    )

print("\nPreprocessor construction integrity: PASSED")
print("No scaler or encoder has been fitted yet.")

LINEAR / COUNT PREPROCESSORS

FULL
----------------------------------------
Numeric inputs: 77
Categorical inputs: 7
Fitted: False

CONTEXT
----------------------------------------
Numeric inputs: 77
Categorical inputs: 3
Fitted: False

NO_SEASON
----------------------------------------
Numeric inputs: 77
Categorical inputs: 6
Fitted: False

Preprocessor construction integrity: PASSED
No scaler or encoder has been fitted yet.


In [14]:
# Fit preprocessing on Fold 1 training data only

fold = temporal_folds_spec[0]

train_start = pd.Timestamp(fold["train_start"])
train_end = pd.Timestamp(fold["train_end"])

val_start = pd.Timestamp(fold["validation_start"])
val_end = pd.Timestamp(fold["validation_end"])


fold1_train = df[
    (df["date"] >= train_start)
    & (df["date"] <= train_end)
].copy()

fold1_val = df[
    (df["date"] >= val_start)
    & (df["date"] <= val_end)
].copy()


# Build a fresh preprocessor.
# Do not reuse a previously fitted transformer.
fold1_preprocessor, fold1_numeric, fold1_categorical = (
    build_linear_preprocessor(
        FEATURE_SETS["full"]
    )
)


# Fit ONLY on historical training matches
X_train_processed = fold1_preprocessor.fit_transform(
    fold1_train[FEATURE_SETS["full"]]
)


# Validation is transform-only
X_val_processed = fold1_preprocessor.transform(
    fold1_val[FEATURE_SETS["full"]]
)


print("FOLD 1 PREPROCESSING TEST")
print("=" * 70)

print(
    "Train period:",
    fold1_train["date"].min().date(),
    "→",
    fold1_train["date"].max().date()
)

print(
    "Validation period:",
    fold1_val["date"].min().date(),
    "→",
    fold1_val["date"].max().date()
)

print("\nRaw input shapes:")
print("Train:", fold1_train[FEATURE_SETS["full"]].shape)
print("Validation:", fold1_val[FEATURE_SETS["full"]].shape)

print("\nProcessed shapes:")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)

print(
    "\nSame processed feature dimension:",
    X_train_processed.shape[1]
    == X_val_processed.shape[1]
)


# Inspect the fitted categorical encoder
onehot = (
    fold1_preprocessor
    .named_transformers_["categorical"]
    .named_steps["onehot"]
)

print("\nCategorical encoding:")
for feature, categories in zip(
    fold1_categorical,
    onehot.categories_
):
    print(
        f"{feature:<20}"
        f"{len(categories):>5} learned categories"
    )


# Confirm transformed matrices contain valid finite values
train_finite = np.isfinite(
    X_train_processed.data
    if hasattr(X_train_processed, "data")
    else X_train_processed
).all()

val_finite = np.isfinite(
    X_val_processed.data
    if hasattr(X_val_processed, "data")
    else X_val_processed
).all()


assert len(fold1_train) == 915
assert len(fold1_val) == 158

assert X_train_processed.shape[0] == 915
assert X_val_processed.shape[0] == 158

assert (
    X_train_processed.shape[1]
    == X_val_processed.shape[1]
)

assert train_finite
assert val_finite


print("\nAll transformed values finite:", True)
print("Fold 1 preprocessing integrity: PASSED")

FOLD 1 PREPROCESSING TEST
Train period: 2002-08-14 → 2018-06-30
Validation period: 2018-08-18 → 2021-06-27

Raw input shapes:
Train: (915, 84)
Validation: (158, 84)

Processed shapes:
Train: (915, 334)
Validation: (158, 334)

Same processed feature dimension: True

Categorical encoding:
season                 16 learned categories
team                    4 learned categories
opponent              203 learned categories
competition            20 learned categories
competition_type        4 learned categories
venue                   3 learned categories
day_of_week             7 learned categories

All transformed values finite: True
Fold 1 preprocessing integrity: PASSED


In [16]:
# Validate preprocessing across every temporal fold

preprocessing_audit_rows = []

for feature_set_name, feature_set in FEATURE_SETS.items():

    for fold in temporal_folds_spec:

        train_start = pd.Timestamp(fold["train_start"])
        train_end = pd.Timestamp(fold["train_end"])

        val_start = pd.Timestamp(fold["validation_start"])
        val_end = pd.Timestamp(fold["validation_end"])

        train = df[
            (df["date"] >= train_start)
            & (df["date"] <= train_end)
        ].copy()

        val = df[
            (df["date"] >= val_start)
            & (df["date"] <= val_end)
        ].copy()


        preprocessor, numeric_features, categorical_features = (
            build_linear_preprocessor(
                feature_set
            )
        )


        X_train_processed = preprocessor.fit_transform(
            train[feature_set]
        )

        X_val_processed = preprocessor.transform(
            val[feature_set]
        )


        train_finite = np.isfinite(
            X_train_processed.data
            if hasattr(X_train_processed, "data")
            else X_train_processed
        ).all()

        val_finite = np.isfinite(
            X_val_processed.data
            if hasattr(X_val_processed, "data")
            else X_val_processed
        ).all()


        preprocessing_audit_rows.append({
            "feature_set": feature_set_name,
            "fold": fold["name"],
            "train_matches": len(train),
            "val_matches": len(val),
            "raw_features": len(feature_set),
            "processed_features": X_train_processed.shape[1],
            "same_dimension": (
                X_train_processed.shape[1]
                == X_val_processed.shape[1]
            ),
            "train_finite": train_finite,
            "val_finite": val_finite,
        })


preprocessing_audit = pd.DataFrame(
    preprocessing_audit_rows
)


print("PREPROCESSING AUDIT — ALL FOLDS")
print("=" * 105)

print(
    preprocessing_audit
    .to_string(index=False)
)


# Integrity checks
assert len(preprocessing_audit) == 12

assert preprocessing_audit[
    "same_dimension"
].all()

assert preprocessing_audit[
    "train_finite"
].all()

assert preprocessing_audit[
    "val_finite"
].all()


print("\nProcessed feature ranges by feature set:")
print("=" * 70)

print(
    preprocessing_audit
    .groupby("feature_set")["processed_features"]
    .agg(["min", "max"])
    .to_string()
)


print("\nAll fold preprocessing checks: PASSED")

PREPROCESSING AUDIT — ALL FOLDS
feature_set   fold  train_matches  val_matches  raw_features  processed_features  same_dimension  train_finite  val_finite
       full Fold 1            915          158            84                 334            True          True        True
       full Fold 2           1073           72            84                 367            True          True        True
       full Fold 3           1145           59            84                 376            True          True        True
       full Fold 4           1204           51            84                 411            True          True        True
    context Fold 1            915          158            80                  91            True          True        True
    context Fold 2           1073           72            80                  91            True          True        True
    context Fold 3           1145           59            80                  91            True          T

In [18]:
# Save preprocessing feature-set specification

import json
from pathlib import Path

ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

preprocessing_spec = {
    "feature_sets": FEATURE_SETS,

    "stable_categoricals": STABLE_CATEGORICALS,

    "identity_categoricals": IDENTITY_CATEGORICALS,

    "preprocessing_strategy": {
        "linear_count_models": {
            "numeric": "StandardScaler",
            "categorical": (
                "OneHotEncoder(handle_unknown='ignore')"
            ),
        },

        "tree_models": {
            "numeric": "no_scaling_required",
            "categorical": (
                "model_specific_encoding"
            ),
        },

        "catboost": {
            "categorical": (
                "native_categorical_handling"
            ),
        },
    },
}

PREPROCESSING_SPEC_PATH = (
    ARTIFACT_DIR / "preprocessing_spec.json"
)

with open(
    PREPROCESSING_SPEC_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        preprocessing_spec,
        f,
        indent=2
    )


print("Saved:", PREPROCESSING_SPEC_PATH)

print("\nFeature sets:")
for name, features in FEATURE_SETS.items():
    print(
        f"{name:<12}"
        f"{len(features):>3} raw features"
    )

print("\nPreprocessing artifact integrity: PASSED")

Saved: ..\artifacts\preprocessing_spec.json

Feature sets:
full         84 raw features
context      80 raw features
no_season    83 raw features

Preprocessing artifact integrity: PASSED
